# 01 · Explore the map

Get to know the reference raster before writing any matching code.

`data/dresden-map.png` is 8192 x 5074 (41.6 MPix), **RGBA** — `cv2.imread` drops alpha by default, which is usually what you want, but be deliberate about it.

In [ ]:
%load_ext autoreload
%autoreload 2

import cv2
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from navmod import matcher, camera, metrics, georef

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA = ROOT / "data"


def show(img, title=None, figsize=(11, 7)):
    """Inline display for OpenCV arrays (handles BGR->RGB and grayscale)."""
    plt.figure(figsize=figsize)
    gray = img.ndim == 2
    plt.imshow(img if gray else cv2.cvtColor(img, cv2.COLOR_BGR2RGB),
               cmap="gray" if gray else None)
    if title:
        plt.title(title)
    plt.axis("off")
    plt.show()


print("ready ·", cv2.__version__)

## Load it

Check dtype, shape and channel count. Decide once whether you work in colour or grayscale and stay consistent — detectors want grayscale anyway.

## Establish scale

What ground distance does the map span? That fixes your GSD (metres/pixel) and makes every later error number physically meaningful. Good first use of `metrics.gsd`.

If it is 5 km across 8192 px, GSD is about 0.61 m/px.

## Look at crops

Pull a few 512x512 windows from different regions — dense city, river, parkland, farmland. These are what your drone will actually see.

## Where are the features?

Run a detector over several crops and count keypoints. Feature-poor regions (water, uniform fields) are where localization will fail, and knowing *which* parts of your map are weak will explain later results. Consider a coarse heatmap of keypoint density over the whole map.

## estimate_pose on Dresden map

Render synthetic frames at known positions, run `estimate_pose`, visualise position error and heading on the map.

In [ ]:
from navmod.matcher import estimate_pose
from navmod.georef import MapGeoref, px_to_latlon, latlon_to_px, distance_m
from navmod.camera import render_frame
from navmod.types import Fix

map_img = cv2.imread(str(DATA / "dresden-map.png"), cv2.IMREAD_COLOR)
print(f"map: {map_img.shape}")

geo = MapGeoref(
    top_left_lat=51.0800079, top_left_lon=13.6911106,
    bottom_right_lat=51.0575373, bottom_right_lon=13.7485796,
    image_width_px=map_img.shape[1], image_height_px=map_img.shape[0],
)
print(f"GSD: {geo.gsd_x:.3f} x {geo.gsd_y:.3f} m/px")

### Single frame — render, match, compare

In [ ]:
# Pick a point — try changing these!
true_px = (4000, 2500)
heading_deg = 30
scale = 1.5  # >1 = higher altitude, sees more ground

frame = render_frame(map_img, *true_px, heading_deg=heading_deg, scale=scale)
fix = estimate_pose(frame, map_img, geo)

true_lat, true_lon = px_to_latlon(geo, *true_px)
if fix.ok:
    err = distance_m(fix.lat, fix.lon, true_lat, true_lon)
    print(f"OK | error: {err:.1f} m | σ: {fix.sigma_m:.2f} m | heading: {np.degrees(fix.heading_rad):.1f}° (true {heading_deg}°)")
else:
    print("NO FIX — matcher failed")

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
axes[0].imshow(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
axes[0].set_title(f"Rendered frame (hdg={heading_deg}°, scale={scale})")
axes[0].axis("off")

# Show neighbourhood on map with true vs estimated position
crop_r = 600
cx, cy = true_px
crop = map_img[max(0,cy-crop_r):cy+crop_r, max(0,cx-crop_r):cx+crop_r]
axes[1].imshow(cv2.cvtColor(crop, cv2.COLOR_BGR2RGB))
axes[1].plot(crop_r, crop_r, "g+", ms=20, mew=3, label="true")
if fix.ok:
    est_px = latlon_to_px(geo, fix.lat, fix.lon)
    dx, dy = est_px[0] - cx + crop_r, est_px[1] - cy + crop_r
    axes[1].plot(dx, dy, "rx", ms=15, mew=3, label=f"est ({err:.0f} m)")
    axes[1].legend(fontsize=11)
axes[1].set_title("Map crop: true (green) vs estimated (red)")
axes[1].axis("off")
plt.tight_layout()
plt.show()

### Grid test — error map across the whole image

In [ ]:
grid_step = 800  # px between test points (smaller = more points, slower)
margin = 300     # stay away from edges
hdg, sc = 0, 1.0

h, w = map_img.shape[:2]
xs = range(margin, w - margin, grid_step)
ys = range(margin, h - margin, grid_step)

results = []  # (px_x, px_y, error_m_or_nan, sigma_m, ok)
for px_x in xs:
    for px_y in ys:
        frame = render_frame(map_img, px_x, px_y, heading_deg=hdg, scale=sc)
        fix = estimate_pose(frame, map_img, geo)
        tlat, tlon = px_to_latlon(geo, px_x, px_y)
        if fix.ok:
            err = distance_m(fix.lat, fix.lon, tlat, tlon)
        else:
            err = np.nan
        results.append((px_x, px_y, err, fix.sigma_m, fix.ok))

results = np.array(results)
n_ok = int(np.sum(results[:, 4]))
errs = results[results[:, 4] == 1, 2]
print(f"{n_ok}/{len(results)} fixes | median err {np.nanmedian(errs):.1f} m | max {np.nanmax(errs):.1f} m")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Error heatmap overlaid on map
thumb = cv2.cvtColor(cv2.resize(map_img, (w//4, h//4)), cv2.COLOR_BGR2RGB)
axes[0].imshow(thumb, extent=[0, w, h, 0])
ok_mask = results[:, 4] == 1
sc_ = axes[0].scatter(results[ok_mask, 0], results[ok_mask, 1],
                       c=results[ok_mask, 2], cmap="RdYlGn_r", s=80,
                       vmin=0, vmax=max(30, np.nanpercentile(errs, 95)),
                       edgecolors="k", linewidths=0.5)
fail_mask = results[:, 4] == 0
if fail_mask.any():
    axes[0].scatter(results[fail_mask, 0], results[fail_mask, 1],
                    c="black", s=80, marker="x", linewidths=2, label="no fix")
    axes[0].legend()
plt.colorbar(sc_, ax=axes[0], label="error (m)")
axes[0].set_title("Position error across map")
axes[0].axis("off")

# Sigma vs actual error — is uncertainty calibrated?
axes[1].scatter(results[ok_mask, 3], results[ok_mask, 2], alpha=0.6, s=40)
lim = max(np.nanmax(errs), np.nanmax(results[ok_mask, 3])) * 1.1
axes[1].plot([0, lim], [0, lim], "k--", lw=1, label="σ = error")
axes[1].set_xlabel("reported σ (m)")
axes[1].set_ylabel("actual error (m)")
axes[1].set_title("Uncertainty calibration")
axes[1].legend()
plt.tight_layout()
plt.show()

### Heading & scale sweep

In [ ]:
cx, cy = 4000, 2500  # fixed position, vary heading & scale

headings = np.arange(0, 360, 15)
scales = [1.0, 1.5, 2.0, 3.0]

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for sc in scales:
    pos_errs, hdg_errs = [], []
    for hdg in headings:
        frame = render_frame(map_img, cx, cy, heading_deg=hdg, scale=sc)
        fix = estimate_pose(frame, map_img, geo)
        tlat, tlon = px_to_latlon(geo, cx, cy)
        if fix.ok:
            pos_errs.append(distance_m(fix.lat, fix.lon, tlat, tlon))
            diff = np.degrees(fix.heading_rad) - hdg
            diff = (diff + 180) % 360 - 180  # wrap to [-180, 180]
            hdg_errs.append(abs(diff))
        else:
            pos_errs.append(np.nan)
            hdg_errs.append(np.nan)
    axes[0].plot(headings, pos_errs, ".-", label=f"scale={sc}")
    axes[1].plot(headings, hdg_errs, ".-", label=f"scale={sc}")

axes[0].set_xlabel("true heading (°)")
axes[0].set_ylabel("position error (m)")
axes[0].set_title("Position error vs heading")
axes[0].legend()
axes[0].set_ylim(bottom=0)

axes[1].set_xlabel("true heading (°)")
axes[1].set_ylabel("heading error (°)")
axes[1].set_title("Heading error vs heading")
axes[1].legend()
axes[1].set_ylim(bottom=0)

plt.tight_layout()
plt.show()